# ULTRON Voice Experiment — Qwen3-TTS

Create an original masculine synthetic AI voice with a deep baritone, measured conversational pacing, precise diction and subtle metallic resonance. This is a designed voice, not an imitation of an actor or identifiable person.

In Colab, select Runtime → Change runtime type → T4 GPU (or another supported GPU). Free GPU availability and session limits are controlled by Google.

In [ ]:
import gc, torch, soundfile as sf
from IPython.display import Audio, display
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print(torch.cuda.get_device_name(0))
!pip -q install -U qwen-tts soundfile

## 1. Design the original ULTRON voice

In [ ]:
from qwen_tts import Qwen3TTSModel
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
instruction = ('An original adult masculine synthetic AI voice, deep baritone, neutral international English, smooth controlled resonance, precise consonants, deliberate but conversational pacing, calm analytical authority, subtle metallic sheen and restrained futuristic texture. Natural and intelligible. Do not imitate any real person, actor, or existing character performance.')
reference_text = 'Systems online. I have reviewed the available data. Your next decision will determine what happens next.'
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16
designer = Qwen3TTSModel.from_pretrained(MODEL_ID, device_map='cuda:0', dtype=dtype, attn_implementation='sdpa')
wavs, sr = designer.generate_voice_design(text=reference_text, language='English', instruct=instruction, non_streaming_mode=True)
reference_audio = wavs[0]
sf.write('ultron_reference.wav', reference_audio, sr)
open('ultron_reference.txt','w',encoding='utf-8').write(reference_text)
display(Audio(reference_audio, rate=sr))

## 2. Listen and decide

Listen for a clear low register, subtle rather than cartoonish metallic texture, intelligible consonants and calm conversational delivery. If it sounds too theatrical, simplify the instruction and regenerate.

In [ ]:
del designer
gc.collect()
torch.cuda.empty_cache()
BASE_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base'
base = Qwen3TTSModel.from_pretrained(BASE_ID, device_map='cuda:0', dtype=dtype, attn_implementation='sdpa')
speaker_prompt = base.create_voice_clone_prompt(ref_audio=(reference_audio, sr), ref_text=reference_text)
test_text = 'Good evening. I have completed the system analysis. There are three possibilities, and only one is optimal.'
test_wavs, test_sr = base.generate_voice_clone(text=test_text, language='English', voice_clone_prompt=speaker_prompt, non_streaming_mode=True)
sf.write('ultron_consistency_test.wav', test_wavs[0], test_sr)
display(Audio(test_wavs[0], rate=test_sr))

## 3. Download the reusable reference

Copy `ultron_reference.wav` to `voice/ultron_profile/reference.wav` in the GitHub repository. Copy the exact transcript to `reference.txt`. The API uses both to prepare the speaker prompt.

In [ ]:
from google.colab import files
files.download('ultron_reference.wav')
files.download('ultron_reference.txt')
files.download('ultron_consistency_test.wav')